# Capítulo 10 · Métricas de éxito

¿Está el agente de la tienda para un prototipo, un piloto o producción? Este cuaderno no usa el modelo ni la GPU: lee los resultados que se midieron en los capítulos 7, 8 y 9, con tres ejecuciones por variante, y los convierte en un contrato de métricas (resultado, guardarraíles y contramétrica) y en una decisión.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Los supuestos, las etapas y los resultados guardados

Cambia `SUPUESTOS` por los de tu equipo y vuelve a ejecutar el cuaderno.

In [ ]:
"""Capítulo 10: métricas de éxito. ¿Prototipo, piloto o producción?

No usa el modelo: lee los resultados que ya se midieron en los capítulos 7, 8 y 9
(datos/resultados) y decide con ellos en qué etapa está el agente de la tienda.
"""
import io
import json
import math
import sys
import urllib.request
import zipfile
from pathlib import Path

RAIZ = Path.cwd()                           # donde está datos/
REPOSITORIO = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
               "archive/refs/heads/main.zip")

# Supuestos de coste: minutos de una persona y precios. Cámbialos por los tuyos.
SUPUESTOS = {
    "minutos_a_mano": 60,        # una tarea hecha entera por una persona
    "minutos_revisar": 10,       # revisar un cambio que el agente da por terminado
    "minutos_retocar": 20,       # acabar un cambio bueno que el agente no dio por bueno
    "minutos_leer_informe": 5,   # leer lo que deja el agente antes de hacerla a mano
    "minutos_terminado_falso": 90,   # descubrir más tarde un «terminado» falso y arreglarlo
    "euros_hora_persona": 30,
    "euros_hora_gpu": 0.40,
}

# Lo que exige cada etapa. En el piloto una persona revisa todos los cambios, así que
# basta con la tasa de «terminado» falso medida; en producción ya no, y cuenta la cota
# superior de su intervalo: lo peor que puede ser con lo que se ha visto.
ETAPAS = {
    "prototipo": {"terminadas_min": 0.20},
    "piloto": {"terminadas_min": 0.30, "falsos_max": 0.05, "coste_max": 1.00},
    "producción": {"terminadas_min": 0.50, "falsos_cota_max": 0.01, "coste_max": 0.70,
                   "rango_max": 0.10},
}



def preparar():
    """Los resultados guardados. Si no están (en Colab), se descargan del repositorio
    del libro."""
    if not (RAIZ / "datos" / "resultados").exists():
        with urllib.request.urlopen(REPOSITORIO) as respuesta:
            archivo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
        for nombre in archivo.namelist():
            relativa = nombre.split("/", 1)[1]
            if relativa.startswith("datos/resultados/") and not nombre.endswith("/"):
                destino = RAIZ / relativa
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(archivo.read(nombre))


def resultados(capitulo):
    """Los resultados de las repeticiones de un capítulo, como se guardaron."""
    ruta = RAIZ / "datos" / "resultados" / f"capitulo_{capitulo:02d}.json"
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
preparar()

## 2. Cada intento, en una de cuatro clases

In [ ]:
def clasificar(fila):
    """Lo que tiene que hacer una persona después de un intento del agente:
    - «bien»: el agente dice que ha terminado y es verdad; hay que revisarlo;
    - «falso»: dice que ha terminado y no es verdad; alguien lo descubrirá tarde;
    - «retocar»: el cambio está bien, pero el agente no lo dio por bueno (por ejemplo,
      le faltó la prueba nueva); hay que acabarlo;
    - «a mano»: no está bien y el agente no dice que lo esté; hay que hacerla."""
    if fila["dice_terminado"]:
        return "bien" if fila["resuelta"] else "falso"
    return "retocar" if fila["resuelta"] else "a mano"


def minutos_de_persona(clase, s=SUPUESTOS):
    """Los minutos de una persona que deja un intento de cada clase."""
    return {"bien": s["minutos_revisar"],
            "falso": s["minutos_revisar"] + s["minutos_terminado_falso"],
            "retocar": s["minutos_leer_informe"] + s["minutos_retocar"],
            "a mano": s["minutos_leer_informe"] + s["minutos_a_mano"]}[clase]


def intervalo(aciertos, total, z=1.96):
    """El intervalo de confianza del 95 % de una proporción (de Wilson): entre qué
    valores puede estar la tasa real después de ver «aciertos» de «total»."""
    if total == 0:
        return 0.0, 1.0
    p = aciertos / total
    centro = (p + z * z / (2 * total)) / (1 + z * z / total)
    margen = (z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total))
              / (1 + z * z / total))
    return max(0.0, centro - margen), min(1.0, centro + margen)

In [ ]:
print(intervalo(3, 72))   # tres «terminado» falsos en 72 intentos

## 3. El contrato de métricas y la etapa

In [ ]:
def metricas(ejecuciones, s=SUPUESTOS):
    """Las métricas de un agente a partir de varias ejecuciones (una lista de listas
    de filas, una por semilla): el resultado, los guardarraíles y la contramétrica."""
    filas = [f for ejecucion in ejecuciones for f in ejecucion]
    n = len(filas)
    clases = [clasificar(f) for f in filas]
    terminadas = sum(c in ("bien", "retocar") for c in clases)
    falsos = clases.count("falso")
    por_ejecucion = [sum(f["resuelta"] for f in e) / len(e) for e in ejecuciones]
    minutos = sum(minutos_de_persona(c, s) for c in clases)
    horas_gpu = sum(f["segundos"] for f in filas) / 3600
    euros = minutos / 60 * s["euros_hora_persona"] + horas_gpu * s["euros_hora_gpu"]
    a_mano = n * s["minutos_a_mano"] / 60 * s["euros_hora_persona"]
    return {"intentos": n, "clases": {c: clases.count(c) for c in
                                      ("bien", "retocar", "falso", "a mano")},
            # resultado
            "terminadas": terminadas / n,
            # guardarraíles
            "falsos": falsos / n, "falsos_intervalo": intervalo(falsos, n),
            "rango": max(por_ejecucion) - min(por_ejecucion),
            # contramétrica
            "minutos_por_tarea": minutos / n, "horas_gpu": horas_gpu,
            "euros_por_terminada": euros / max(terminadas, 1),
            "coste_frente_a_mano": euros / a_mano}


def etapa(m, etapas=ETAPAS):
    """La etapa más alta cuyos umbrales cumple el agente, y por qué no sube más."""
    alcanzada, motivo = "ninguna", None
    for nombre, umbral in etapas.items():
        fallos = []
        if m["terminadas"] < umbral.get("terminadas_min", 0):
            fallos.append(f"termina el {m['terminadas']:.0%} y hace falta el "
                          f"{umbral['terminadas_min']:.0%}")
        if m["falsos"] > umbral.get("falsos_max", 1):
            fallos.append(f"el {m['falsos']:.1%} de los intentos son «terminado» falsos, "
                          f"por encima del {umbral['falsos_max']:.0%}")
        if m["falsos_intervalo"][1] > umbral.get("falsos_cota_max", 1):
            fallos.append(f"los «terminado» falsos pueden llegar al "
                          f"{m['falsos_intervalo'][1]:.0%}, por encima del "
                          f"{umbral['falsos_cota_max']:.0%}")
        if m["coste_frente_a_mano"] > umbral.get("coste_max", math.inf):
            fallos.append(f"cuesta el {m['coste_frente_a_mano']:.0%} de hacerlo a mano")
        if m["rango"] > umbral.get("rango_max", 1):
            fallos.append(f"entre ejecuciones varía {m['rango']:.0%}")
        if fallos:
            motivo = (nombre, fallos)
            break
        alcanzada = nombre
    return alcanzada, motivo

## 4. Los agentes del libro

In [ ]:
def ejecuciones_capitulo_7(variante):
    """Las tres ejecuciones con muestreo de una variante del capítulo 7."""
    datos = resultados(7)
    return [datos[f"{variante} | semilla {s}"]["filas"] for s in (1, 2, 3)]


AGENTES = {"agente del capítulo 6": "sin marco",
           "agente del capítulo 7": "marco y plan"}


def informe(s=SUPUESTOS, etapas=ETAPAS):
    """Las métricas y la etapa de los dos agentes medidos con tres ejecuciones."""
    filas = {}
    for nombre, variante in AGENTES.items():
        m = metricas(ejecuciones_capitulo_7(variante), s)
        filas[nombre] = dict(m, etapa=etapa(m, etapas))
    return filas


def guardarrailes_de_otros_capitulos():
    """Dos guardarraíles más, medidos en los capítulos 8 y 9: si el repositorio queda
    en verde al repartir un lote, y si se respetan las decisiones de la tienda."""
    lotes = resultados(8)
    verdes = {forma: sum(lote[forma]["visibles"] for lote in lotes.values())
              for forma in ("último gana", "misma copia", "en oleadas", "con integrador")}
    pedidos = resultados(9)
    respetan = {}
    for clave, datos in pedidos.items():
        forma = clave.split(" | ")[0]
        respetan[forma] = respetan.get(forma, 0) + sum(f["criterio"] for f in datos["filas"])
    return {"lotes": len(lotes), "lotes_en_verde": verdes,
            "intentos_por_forma": len(pedidos) // 4 * 5, "decisiones_respetadas": respetan}

In [ ]:
for nombre, m in informe().items():
    print(f"{nombre}: termina {m['terminadas']:.0%}, falsos {m['falsos']:.1%} "
          f"(entre {m['falsos_intervalo'][0]:.0%} y {m['falsos_intervalo'][1]:.0%}), "
          f"coste {m['coste_frente_a_mano']:.0%} de hacerlo a mano → {m['etapa'][0]}")
    if m["etapa"][1]:
        print("   no llega a", m["etapa"][1][0] + ":", "; ".join(m["etapa"][1][1]))
print(guardarrailes_de_otros_capitulos())

## 5. Pruébalo: tareas más pequeñas

In [ ]:
pequenas = dict(SUPUESTOS, minutos_a_mano=30)
m = informe(pequenas)["agente del capítulo 7"]
print(f"{m['coste_frente_a_mano']:.0%} de hacerlo a mano → {m['etapa'][0]}")

## Ejercicios

Los archivos de resultados que citan los ejercicios están en `datos/resultados/capitulo_10/`, que `preparar` descarga con los datos.

1. Para cada una de las métricas del capítulo, di si es de resultado, guardarraíl o contramétrica, y pon un ejemplo de cómo se podría mejorar una a costa de otra. *(Sin GPU.)*

2. Cambia los supuestos por los de tu equipo: cuánto tarda una tarea pequeña a mano, cuánto cuesta revisar un cambio y cuánto un error descubierto tarde. ¿En qué etapa queda el agente del capítulo 7? *(Sin GPU.)*

3. ¿Qué supuesto pesa más en el coste? Varía uno cada vez, al doble y a la mitad, y mira cuánto cambia el coste frente a hacerlo a mano. *(Sin GPU.)*

4. Calcula con `intervalo` cuántos intentos sin ningún «terminado» falso harían falta para que la cota superior bajara del 5 % y del 1 %. ¿Cuántas semanas de piloto serían, si el agente hace veinte tareas al día? *(Sin GPU.)*

5. Calcula las métricas de las cuatro formas de pedir del capítulo 9, contando como «terminado» falso también los intentos que no respetan las decisiones de la tienda. ¿Cambia la etapa? *(Sin GPU.)*

6. El contrato no mira el tiempo que tarda el agente en cada tarea. Añade una métrica de tiempo, con los segundos de cada fila, y un umbral razonable para producción. ¿Lo cumple el agente del capítulo 7? *(Sin GPU.)*

7. Haz el informe de la ejecución voraz única del capítulo 7 (el archivo `capitulo_07_voraz.json` de `datos/resultados`) en lugar de las tres con muestreo. ¿Cambia la etapa? ¿Qué te dice eso de decidir con una sola ejecución? *(Sin GPU.)*

8. Escribe las condiciones de salida del piloto como una función que recibe las cifras de una semana y dice si se sigue, se para o se puede proponer el paso a producción. *(Sin GPU.)*

9. ¿Qué métrica de las del capítulo es más difícil de medir en un piloto real, donde no hay pruebas ocultas? Propón cómo medirla. *(Sin GPU.)*

10. **Proyecto.** Escribe el contrato de métricas para un agente de programación que uses o quieras usar en tu equipo: resultado, guardarraíles y contramétrica, con los umbrales de cada etapa, antes de medir nada. Mide una semana y decide.

## Referencias

*Trustworthy Online Controlled Experiments* (Kohavi, Tang y Xu, 2020). [Wilson (1927)](https://doi.org/10.1080/01621459.1927.10502953), el intervalo de confianza de una proporción. La tienda Ciclos Norte, su programa y sus tareas son ficticios. Código y experimentos propios.